# Outputs

**Outputs** written to `prepared/`
- `participants.csv`, one row per participant with the resolved tool assignment
- `understanding_long.csv`, one row per answered task page
- `forced_choice_long.csv`, one row per participant and slider
- `performance_long.csv` and `performance_item_long.csv`, participant- and item-level objective task scores
- `*_plot.csv`, descriptive means and sample sizes for the article plots
- `plot_data.json`, versionable aggregate values used by the Typst figures

In [200]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PREPARED_DIR = Path("prepared")
PREPARED_DIR.mkdir(exist_ok=True)

csv_file = "Regression+Lab+Study+-+Questionnaire+2026+version+-+lab-version-2-scenarios_September+21,+2026_14.54.csv"

export = pd.read_csv(csv_file, dtype=str, keep_default_na=False, encoding="utf-8-sig")
question_texts = export.loc[export["StartDate"] == "Start Date"].iloc[0].to_dict()
item_columns = [
    column
    for column in export.columns
    if not column.startswith("Last Seen Question IDs_")
]
responses = export[export["ResponseId"].str.startswith("R_")][item_columns].reset_index(
    drop=True
)

print(
    f"{len(responses)} responses, {len(item_columns)} usable columns (2 metadata rows and the routing columns dropped)"
)

111 responses, 135 usable columns (2 metadata rows and the routing columns dropped)


## Analytic sample

In [201]:
CONSENT_TEXT = "Ich stimme der Verwendung meiner Daten für die Forschung zu."
TEST_PATTERN = r"test|florin|arthi"

excluded = (
    responses["Status"].eq("Survey Preview")
    | responses["DistributionChannel"].eq("preview")
    | responses["Pseudonym"].str.lower().str.contains(TEST_PATTERN)
    | responses["Computer-ID "].str.lower().eq("test")
    | ~responses["Consent "].eq(CONSENT_TEXT)
    | ~responses["Finished"].eq("True")
    # also exclude any row that has a cell that contains "datensatz löschen, fehler in der studie"
    | responses.apply(
        lambda row: row.astype(str)
        .str.contains("datensatz löschen, fehler in der studie", case=False)
        .any(),
        axis=1,
    )
)
answered = (responses[item_columns] != "").sum(axis=1)

participants = (
    responses.loc[~excluded]
    .assign(answered=answered[~excluded])
    .sort_values(
        ["Pseudonym", "answered", "RecordedDate"], ascending=[True, False, False]
    )
    .drop_duplicates("Pseudonym")
    .reset_index(drop=True)
)
print(f"{len(responses)} responses -> {len(participants)} participants")

111 responses -> 101 participants


## Resolved tool per scenario

In [202]:
TOOL_ALIASES = {
    "Spezielles Tool": "CLM tool",
    "tool": "CLM tool",
    "RStudio": "RStudio output",
    "chatgpt": "RStudio output",
}


def assigned_tool(row, scenario):
    """Tool this participant used for one scenario, read from the counterbalancing fields."""
    if row["FirstScenario"] == scenario:
        return TOOL_ALIASES.get(row["FirstMethod"])
    if row["SecondScenario"] == scenario:
        return TOOL_ALIASES.get(row["SecondMethod"])
    return None


for scenario in ("simple", "complex"):
    participants[f"tool_{scenario}"] = [
        assigned_tool(row, scenario) for _, row in participants.iterrows()
    ]
participants["assignment_ok"] = (
    participants["tool_simple"].isin(set(TOOL_ALIASES.values()))
    & participants["tool_complex"].isin(set(TOOL_ALIASES.values()))
    & participants["tool_simple"].ne(participants["tool_complex"])
)
print(
    participants["assignment_ok"]
    .value_counts()
    .rename_axis("tool assignment resolved")
    .to_string()
)

tool assignment resolved
True     96
False     5


## Self rated understanding

In [203]:
def to_number(value):
    """Parse answers such as '9', '5- Mäßig' or '10 - Sehr' into a float."""
    if not isinstance(value, str) or not value.strip():
        return np.nan
    match = re.search(r"-?\d+(?:[.,]\d+)?", value)
    return np.nan if match is None else float(match.group().replace(",", "."))


UNDERSTANDING_QID = {
    ("CLM tool", "simple"): "Q220",
    ("CLM tool", "complex"): "Q210",
    ("RStudio output", "simple"): "Q209",
    ("RStudio output", "complex"): "Q215",
}

rows = []
for _, record in participants.iterrows():
    for (tool, scenario), question_id in UNDERSTANDING_QID.items():
        if record[f"tool_{scenario}"] != tool:
            continue
        score = to_number(record[question_id])
        if not np.isnan(score):
            rows.append(
                {
                    "participant": record["Pseudonym"],
                    "tool": tool,
                    "scenario": scenario,
                    "score": score,
                    "assignment_ok": record["assignment_ok"],
                    "source_qid": question_id,
                }
            )
understanding_long = pd.DataFrame(rows)

print(
    understanding_long.groupby(["tool", "scenario"])["score"]
    .agg(pages="size", mean="mean", median="median", min="min", max="max")
    .round(2)
    .to_string()
)

                         pages  mean  median  min   max
tool           scenario                                
CLM tool       complex      44  6.68     7.0  3.0  10.0
               simple       51  7.94     8.0  5.0  10.0
RStudio output complex      51  5.00     5.0  1.0   8.0
               simple       44  7.16     7.0  2.0  10.0


## Objective task performance

Score the closed interpretation questions using the answer key derived from `simple.R` and `complex.R`. The checkbox group counts as one question and is correct only when all correct options, and no incorrect options, were selected.

In [204]:
GPA_ANSWER = (
    "Ein höherer GPA ist tendenziell mit einer höheren Bewerbungswahrscheinlichkeit "
    "assoziiert. Der Effekt ist statistisch signifikant."
)
SERVICE_ANSWER = (
    "Ein Pflichtkurs erhält tendenziell niedrigere Bewertungen. "
    "Der Unterschied ist statistisch signifikant."
)
INTERACTION_ANSWER = (
    "Es gibt keine statistische Evidenz, dass sich der Pflicht-Effekt zwischen "
    "Early-Stage und Late-Stage Studierenden unterscheidet. Der Unterschied ist "
    "nicht signifikant (α = 0.05)."
)
RANDOM_EFFECT_ANSWER = (
    "Das durchschnittliche Bewertungsniveau, das Dozierende erhalten, unterscheidet "
    "sich stärker zwischen einzelnen Dozierenden als die Bewertungstendenz, mit der "
    "einzelne Studierende bewerten."
)


def simple_key(prefix):
    return {
        "checkbox_item": "pared and public effects",
        "checkboxes": [f"{prefix}_{number}" for number in range(1, 7)],
        "selected": {f"{prefix}_1", f"{prefix}_5"},
    }


def complex_key(prefix):
    suffixes = (1, 2, 8, 3, 9, 10, 11, 12)
    return {
        "checkbox_item": "service effects by study stage",
        "checkboxes": [f"{prefix}_{suffix}" for suffix in suffixes],
        "selected": {f"{prefix}_1", f"{prefix}_10"},
    }


ANSWER_KEY = {
    ("CLM tool", "simple"): {
        **simple_key("Q199"),
        "single": {
            "Q200": ("number of observations", "400"),
            "Q201": ("GPA effect", GPA_ANSWER),
        },
    },
    ("RStudio output", "simple"): {
        **simple_key("Q132"),
        "single": {
            "Q133": ("number of observations", "400"),
            "Q134": ("GPA effect", GPA_ANSWER),
        },
    },
    ("RStudio output", "complex"): {
        **complex_key("Q120"),
        "single": {
            "Q121": ("number of students", "299"),
            "Q122": ("early-stage service effect", SERVICE_ANSWER),
            "Q123": ("service by study-stage interaction", INTERACTION_ANSWER),
            "Q124": ("larger random effect", RANDOM_EFFECT_ANSWER),
        },
    },
    ("CLM tool", "complex"): {
        **complex_key("Q167"),
        "single": {
            "Q168": ("number of students", "299"),
            "Q169": ("early-stage service effect", SERVICE_ANSWER),
            "Q170": ("service by study-stage interaction", INTERACTION_ANSWER),
            "Q171": ("larger random effect", RANDOM_EFFECT_ANSWER),
        },
    },
}


def clean_answer(value):
    return value.strip() if isinstance(value, str) else ""


item_rows = []
for _, record in participants.iterrows():
    for scenario in ("simple", "complex"):
        tool = record[f"tool_{scenario}"]
        key = ANSWER_KEY.get((tool, scenario))
        if key is None:
            continue

        question_columns = key["checkboxes"] + list(key["single"])
        if not any(clean_answer(record[column]) for column in question_columns):
            continue

        selected = {
            column for column in key["checkboxes"] if clean_answer(record[column])
        }
        item_scores = [
            (key["checkbox_item"], int(selected == key["selected"]))
        ]
        item_scores += [
            (item, int(clean_answer(record[column]) == answer))
            for column, (item, answer) in key["single"].items()
        ]
        for item, correct in item_scores:
            item_rows.append(
                {
                    "participant": record["Pseudonym"],
                    "tool": tool,
                    "scenario": scenario,
                    "item": item,
                    "correct": correct,
                    "assignment_ok": record["assignment_ok"],
                }
            )

performance_item_long = pd.DataFrame(item_rows)
performance_long = (
    performance_item_long.groupby(
        ["participant", "tool", "scenario", "assignment_ok"], as_index=False
    )
    .agg(correct=("correct", "sum"), questions=("correct", "size"))
)
performance_long["accuracy"] = (
    performance_long["correct"] / performance_long["questions"]
)
performance_summary = (
    performance_long[performance_long["assignment_ok"]]
    .groupby(["scenario", "tool"])
    .agg(
        participants=("participant", "nunique"),
        mean_correct=("correct", "mean"),
        questions=("questions", "first"),
        percent_correct=("accuracy", lambda values: 100 * values.mean()),
    )
    .round(2)
)
print(performance_summary.to_string())

performance_item_summary = (
    performance_item_long[performance_item_long["assignment_ok"]]
    .groupby(["scenario", "item", "tool"])
    .agg(
        answers=("correct", "size"),
        percent_correct=("correct", lambda values: 100 * values.mean()),
    )
    .round(2)
)
print("\nAccuracy by question:")
print(performance_item_summary.to_string())

item_comparison = (
    performance_item_summary["percent_correct"]
    .unstack("tool")
    .rename(
        columns={
            "CLM tool": "clm_percent",
            "RStudio output": "rstudio_percent",
        }
    )
)
item_comparison["clm_minus_rstudio"] = (
    item_comparison["clm_percent"] - item_comparison["rstudio_percent"]
)
item_comparison["higher_accuracy"] = np.select(
    [
        item_comparison["clm_minus_rstudio"] > 0,
        item_comparison["clm_minus_rstudio"] < 0,
    ],
    ["CLM tool", "RStudio output"],
    default="tie",
)
print("\nQuestion comparison:")
print(item_comparison.round(2).to_string())

print("\nHigher mean accuracy by scenario:")
summary_rows = performance_summary.reset_index()
for scenario, group in summary_rows.groupby("scenario"):
    best = group.loc[group["percent_correct"].idxmax()]
    print(f"{scenario}: {best['tool']} ({best['percent_correct']:.1f}%)")

                         participants  mean_correct  questions  percent_correct
scenario tool                                                                  
complex  CLM tool                  44          2.93          5            58.64
         RStudio output            51          2.27          5            45.49
simple   CLM tool                  51          2.43          3            81.05
         RStudio output            44          2.52          3            84.09

Accuracy by question:
                                                            answers  percent_correct
scenario item                               tool                                    
complex  early-stage service effect         CLM tool             44            68.18
                                            RStudio output       51            60.78
         larger random effect               CLM tool             44            84.09
                                            RStudio output       51     

## Forced choice sliders

The raw slider runs from the first tool (0) to the second tool (10). Normalize its direction so that 0 always represents the RStudio output and 10 always represents the CLM tool.

In [205]:
FORCED_CHOICE_ITEMS = {
    "TrustTool_1": "best supported during interpretation",
    "Q226_1": "more usable",
    "Q227_1": "would prefer for interpreting a regression",
}

mapping = {
    "TrustTool_1": "Unterstützung",
    "Q226_1": "Benutzerfreundlichkeit",
    "Q227_1": "Präferenz",
}

forced_choice_long = (
    pd.DataFrame(
        {
            "participant": participants["Pseudonym"].to_numpy(),
            "first_tool": participants["FirstMethod"].map(TOOL_ALIASES).to_numpy(),
            "second_tool": participants["SecondMethod"].map(TOOL_ALIASES).to_numpy(),
            **{
                mapping[column]: participants[column].map(to_number).to_numpy()
                for column in FORCED_CHOICE_ITEMS
            },
        }
    )
    .melt(
        id_vars=["participant", "first_tool", "second_tool"],
        var_name="item",
        value_name="raw_slider_value",
    )
    .dropna(subset=["raw_slider_value"])
)

forced_choice_long["raw_slider_value"] = forced_choice_long["raw_slider_value"].astype(
    int
)
clm_is_second = forced_choice_long["first_tool"].eq(
    "RStudio output"
) & forced_choice_long["second_tool"].eq("CLM tool")
clm_is_first = forced_choice_long["first_tool"].eq("CLM tool") & forced_choice_long[
    "second_tool"
].eq("RStudio output")
forced_choice_long["slider_value"] = np.select(
    [clm_is_second, clm_is_first],
    [
        forced_choice_long["raw_slider_value"],
        10 - forced_choice_long["raw_slider_value"],
    ],
    default=np.nan,
)
forced_choice_long = forced_choice_long.dropna(subset=["slider_value"])
forced_choice_long["slider_value"] = forced_choice_long["slider_value"].astype(int)
forced_choice_long["lean_towards_clm"] = forced_choice_long["slider_value"] - 5

for column in FORCED_CHOICE_ITEMS:
    print(f"{mapping[column]}: {question_texts[column]}")

print()

print(
    forced_choice_long.groupby("item")["slider_value"]
    .agg(
        answers="size",
        mean="mean",
        median="median",
        min="min",
        max="max",
    )
    .round(2)
    .to_string()
)

Unterstützung: Welches Tool hat Sie bei der Interpretation am besten unterstützt? - Unterstützung
Benutzerfreundlichkeit: Welches Tool empfinden Sie als benutzerfreundlicher? - Benutzerfreundlichkeit
Präferenz: Welches Tool würden Sie vorziehen, wenn Sie eine Regression interpretieren müssten? - Präferenz

                        answers  mean  median  min  max
item                                                   
Benutzerfreundlichkeit       95  8.62    10.0    1   10
Präferenz                    95  7.86     9.0    0   10
Unterstützung                95  7.87     8.0    1   10


## General self-rated understanding of regression

For the 95-row valid analysis sample, the observed answers are 5 "wenig verstanden", 78 "etwas verstanden", and 12 "Vollständig verstanden". The next cell calculates percentages and plots the distribution. The sample is defined by a resolved tool assignment and a response to all three forced-choice items.


In [ ]:
GENERAL_UNDERSTANDING_QUESTION = (
    "Wie sehr, glauben Sie, haben Sie das Thema Regression innerhalb der Vorlesung "
    "oder im Selbststudium verstanden?"
)
GENERAL_UNDERSTANDING_QID = next(
    column
    for column, text in question_texts.items()
    if text == GENERAL_UNDERSTANDING_QUESTION
)

complete_slider_participants = set(
    forced_choice_long.groupby("participant")["item"]
    .nunique()
    .loc[lambda item_counts: item_counts == len(FORCED_CHOICE_ITEMS)]
    .index
)
valid_respondents = participants.loc[
    participants["assignment_ok"]
    & participants["Pseudonym"].isin(complete_slider_participants),
    ["Pseudonym", "ResponseId"],
]
understanding_answers = valid_respondents.merge(
    responses[["ResponseId", GENERAL_UNDERSTANDING_QID]],
    on="ResponseId",
    validate="one_to_one",
).rename(columns={GENERAL_UNDERSTANDING_QID: "response"})
understanding_answers["response"] = understanding_answers["response"].str.strip()
understanding_answers = understanding_answers.loc[
    understanding_answers["response"].ne("")
]

response_order = [
    "wenig verstanden",
    "etwas verstanden",
    "Vollständig verstanden",
]
response_order.extend(
    sorted(set(understanding_answers["response"]) - set(response_order))
)
response_counts = understanding_answers["response"].value_counts().reindex(
    response_order, fill_value=0
)
understanding_distribution = pd.DataFrame(
    {
        "responses": response_counts,
        "percent": (100 * response_counts / response_counts.sum()).round(1),
    }
)
print(f"Valid responses: {len(understanding_answers)}")
display(understanding_distribution)

ax = understanding_distribution["responses"].plot.barh(
    figsize=(8, 3.4), color="#4C78A8", width=0.68
)
ax.invert_yaxis()
ax.set_title("Self-rated understanding of regression")
ax.set_xlabel("Number of responses")
ax.set_ylabel("")
max_count = max(understanding_distribution["responses"].max(), 1)
ax.set_xlim(0, max_count * 1.28)
for bar, (_, row) in zip(ax.patches, understanding_distribution.iterrows()):
    ax.text(
        row["responses"] + max_count * 0.025,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['responses'])} ({row['percent']:.1f}%)",
        va="center",
    )
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_axisbelow(True)
ax.xaxis.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


## Export

In [206]:
participants_export = participants[
    [
        "Pseudonym",
        "ResponseId",
        "RecordedDate",
        "Duration (in seconds)",
        "FirstScenario",
        "SecondScenario",
        "tool_simple",
        "tool_complex",
        "assignment_ok",
        "answered",
    ]
]

def mean_summary(data, group_columns, value_column, *, scale=1):
    """Return descriptive group means and numbers of participants."""
    rows = []
    for keys, group in data.groupby(group_columns, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        values = group[value_column].dropna().to_numpy(dtype=float) * scale
        rows.append(
            {
                **dict(zip(group_columns, keys)),
                "n": len(values),
                "mean": values.mean(),
            }
        )
    return pd.DataFrame(rows).round(2)


performance_plot = mean_summary(
    performance_long[performance_long["assignment_ok"]],
    ["scenario", "tool"],
    "accuracy",
    scale=100,
)
understanding_plot = mean_summary(
    understanding_long[understanding_long["assignment_ok"]],
    ["scenario", "tool"],
    "score",
)
preference_plot = mean_summary(
    forced_choice_long, ["item"], "slider_value"
)

participants_export.to_csv(PREPARED_DIR / "participants.csv", index=False)
understanding_long.to_csv(PREPARED_DIR / "understanding_long.csv", index=False)
forced_choice_long.to_csv(PREPARED_DIR / "forced_choice_long.csv", index=False)
performance_long.to_csv(PREPARED_DIR / "performance_long.csv", index=False)
performance_item_long.to_csv(
    PREPARED_DIR / "performance_item_long.csv", index=False
)
performance_plot.to_csv(PREPARED_DIR / "performance_plot.csv", index=False)
understanding_plot.to_csv(PREPARED_DIR / "understanding_plot.csv", index=False)
preference_plot.to_csv(PREPARED_DIR / "preference_plot.csv", index=False)
plot_data = {
    "performance": performance_plot.to_dict(orient="records"),
    "understanding": understanding_plot.to_dict(orient="records"),
    "preference": preference_plot.to_dict(orient="records"),
}
(PREPARED_DIR / "plot_data.json").write_text(
    json.dumps(plot_data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

assert understanding_long["score"].between(0, 10).all()
assert forced_choice_long["slider_value"].between(0, 10).all()
assert understanding_long["assignment_ok"].isin([True, False]).all()
assert not understanding_long.duplicated(["participant", "tool", "scenario"]).any()
assert not forced_choice_long.duplicated(["participant", "item"]).any()
assert set(understanding_long["participant"]) <= set(participants_export["Pseudonym"])
assert set(forced_choice_long["participant"]) <= set(participants_export["Pseudonym"])
print("checks passed")
for path in sorted(PREPARED_DIR.glob("*.csv")):
    print(f"  {path}  {path.stat().st_size / 1024:.0f} KiB")

checks passed
  prepared/forced_choice_long.csv  18 KiB
  prepared/participants.csv  11 KiB
  prepared/performance_item_long.csv  50 KiB
  prepared/performance_long.csv  10 KiB
  prepared/performance_plot.csv  0 KiB
  prepared/plot_data.json  1 KiB
  prepared/preference_plot.csv  0 KiB
  prepared/understanding_long.csv  9 KiB
  prepared/understanding_plot.csv  0 KiB
